# 第一步：啟動系統
- 這一步是安裝ComfyUI和常用插件
- 運行時間大約是1-5分鐘，取決於你的網絡連結情況

Git clone the repo and install the requirements. (ignore the pip errors about protobuf)

USE_GOOGLE_DRIVE若有需要保存才需勾選

In [ ]:
# #@title Environment Setup

from pathlib import Path

OPTIONS = {}

USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
USE_COMFYUI_MANAGER = True  #@param {type:"boolean"}
INSTALL_CUSTOM_NODES_DEPENDENCIES = True  #@param {type:"boolean"}
OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI
OPTIONS['USE_COMFYUI_MANAGER'] = USE_COMFYUI_MANAGER
OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES'] = INSTALL_CUSTOM_NODES_DEPENDENCIES

current_dir = !pwd
WORKSPACE = "/content/ComfyUI"

if OPTIONS['USE_GOOGLE_DRIVE']:
    !echo "Mounting Google Drive..."
    %cd /

    from google.colab import drive
    drive.mount('/content/drive')

    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
    %cd /content/drive/MyDrive

![ ! -d $WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-

  # Correction of the issue of permissions being deleted on Google Drive.
  ![ -f ".ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/nightly/windows_base_files/run_nvidia_gpu.bat" ] && chmod 755 .ci/nightly/windows_base_files/run_nvidia_gpu.bat
  ![ -f ".ci/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat
  ![ -f ".ci/update_windows/update.py" ] && chmod 755 .ci/update_windows/update.py
  ![ -f ".ci/update_windows/update_comfyui.bat" ] && chmod 755 .ci/update_windows/update_comfyui.bat
  ![ -f ".ci/update_windows/README_VERY_IMPORTANT.txt" ] && chmod 755 .ci/update_windows/README_VERY_IMPORTANT.txt
  ![ -f ".ci/update_windows/run_cpu.bat" ] && chmod 755 .ci/update_windows/run_cpu.bat
  ![ -f ".ci/update_windows/run_nvidia_gpu.bat" ] && chmod 755 .ci/update_windows/run_nvidia_gpu.bat

  !git pull

!echo -= Install dependencies =-
#!pip3 install accelerate
#!pip3 install einops transformers>=4.28.1 safetensors>=0.4.2 aiohttp pyyaml Pillow scipy tqdm psutil tokenizers>=0.13.3
#!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
#!pip3 install torchsde
#!pip3 install kornia>=0.7.1 spandrel soundfile sentencepiece
!pip3 install -r requirements.txt
!pip3 install -r manager_requirements.txt

if OPTIONS['USE_COMFYUI_MANAGER']:
  %cd custom_nodes
  ![ -f "ComfyUI-Manager/check.sh" ] && chmod 755 ComfyUI-Manager/check.sh
  ![ -f "ComfyUI-Manager/scan.sh" ] && chmod 755 ComfyUI-Manager/scan.sh
  ![ -f "ComfyUI-Manager/node_db/dev/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/dev/scan.sh
  ![ -f "ComfyUI-Manager/node_db/tutorial/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/tutorial/scan.sh
  ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh
  ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-win.bat" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-win.bat
  ![ ! -d ComfyUI-Manager ] && echo -= Initial setup ComfyUI-Manager =- && git clone https://github.com/ltdrdata/ComfyUI-Manager

  !git clone https://github.com/city96/ComfyUI-GGUF.git
  !git clone https://github.com/kijai/ComfyUI-KJNodes.git
  !git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
  %cd ComfyUI-Manager
  !git pull

%cd $WORKSPACE

if OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES']:
  !echo -= Install custom nodes dependencies =-
  !pip install GitPython
  !python custom_nodes/ComfyUI-Manager/cm-cli.py restore-dependencies


# 第二步： 下載模型
- 下載模型(預設所需連結如下程式可自行添加)
- 如何下載任意模型和風格模型?
1. 連結Civitai帳號，透過API直接下載網站上的模型
2. 一般情況模型放在checkpoints文件夾，風格模型放在loras文件夾
3. 對於一般的模型和lora, 你需要替換下面三個位置的信息，他們分別是:
   - url: 這個文件的下載地址(在Civitai對該模型的下載按鈕點右鍵->複製連結網址)
   - folder: 這個文件應該存放的文件夾名字，一般模型放在checkpoints, 風格模型放在loras
   - name: 給它起一個名字，名字的格式是 xxx.safetensors。其中， xxx只能使用英文、數字和下劃線。不同的模型需要有不同的名字，否則文件會被覆蓋
   - 請參考這些命名方式：sd35_large_fp8.safetensors， majic.safetensors， anylora.safetensors

連結API

In [ ]:
# 用你的 civitai token 替換 XYZ
CIVITAI_TOEKN = "XYZ"

print("***** 你的token已經被應用到臨時變量中✅！")
print("civitai token: ", CIVITAI_TOEKN)

下載模型Function(預設下載VAE，但是否使用取決於下載模型是否已包含VAE)

In [ ]:
BASE = "/content/ComfyUI/models"
def download_from_civitai(url, folder, name):
    tok = CIVITAI_TOEKN
    model_url = f"{url}&token={tok}"
    save_path = BASE + "/" + folder + "/" + name
    get_ipython().system(f'wget -O "{save_path}" "{model_url}"')
url = "https://civitai.com/api/download/models/311162?type=Model&format=SafeTensor"
folder = "vae"
name = "vae-ft-mse-840000-ema-pruned.safetensors"
download_from_civitai(url, folder, name)

### 第1個模型： Animagine XL 4.0 模型, https://civitai.com/models/1188071?modelVersionId=1337429
- 類型：sdxl
- 文件大小： 6.46G

In [ ]:
url = "https://civitai.com/api/download/models/1494740?type=Model&format=SafeTensor&size=pruned&fp=fp16"
folder = "checkpoints"
name = "Obsession.safetensors"
download_from_civitai(url, folder, name)

### Lora模型1：test-lora，https://civitai.com/models/1209474/a4-test?modelVersionId=1373925
- 類型: sdxl lora 模型
- 文件大小：0.1G

In [ ]:
url = "https://civitai.com/api/download/models/1373925?type=Model&format=SafeTensor"
folder = "loras"
name = "mix-a4-v2.safetensors"
download_from_civitai(url, folder, name)

### Run ComfyUI




In [ ]:
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()

    print("\nComfyUI 載入，正在啟動 localtunnel...\n")
    try:
        req = urllib.request.Request('https://ipv4.icanhazip.com')
        ip = urllib.request.urlopen(req).read().decode('utf8').strip()
        print("=========================================================")
        print(f"請在網頁的「Endpoint IP」欄位中輸入這個密碼： {ip}")
        print("=========================================================\n")
    except Exception as e:
        print("無法獲取 IP:", e)

    p = subprocess.Popen(["lt", "--port", str(port)], stdout=subprocess.PIPE)
    for line in p.stdout:
        print("ComfyUI 網址:", line.decode().strip())

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python /content/ComfyUI/main.py --enable-manager --dont-print-server